# KEIBA AI — 重みチューニング & キャリブレーション

**実行タイミング**: 初回 + 月1回（新しいレースデータが50件以上増えたとき）

| セル | 内容 |
|---|---|
| セル1 | 環境設定・Driveマウント |
| セル2 | src/ クローン（GitHub → Drive） |
| セル3 | エンジン初期化 |
| セル4 | **重みチューニング**（〜2分） |
| セル5 | **キャリブレーション** |
| セル6 | **乖離分析・診断** |

## セル1: 環境設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys

BASE_DIR = '/content/drive/MyDrive/keiba_ai'
os.makedirs(f'{BASE_DIR}/data', exist_ok=True)
print(f'BASE_DIR: {BASE_DIR}')

# history.db または keiba.db の存在確認
for db in ['history.db', 'keiba.db']:
    path = f'{BASE_DIR}/data/{db}'
    print(f'{db}: {"✅ あり" if os.path.exists(path) else "❌ なし"}')

## セル2: src/ クローン（GitHub → Drive）
初回のみ実行。すでに `/content/drive/MyDrive/keiba_ai/src/` があればスキップ可。

In [ ]:
import os, subprocess, shutil

REPO    = 'https://github.com/hanagenuku/keiba_ai.git'
BRANCH  = 'main'  # PRマージ前なら 'claude/review-drive-document-ZehuT' に変更
SRC_DIR = f'{BASE_DIR}/src'

# 毎回最新版を取得（古いcloneがあっても上書き）
if os.path.exists('/tmp/keiba_clone'):
    shutil.rmtree('/tmp/keiba_clone')

subprocess.run(['git','clone','--no-checkout','--depth=1',
                '--branch', BRANCH, REPO, '/tmp/keiba_clone'], check=True)
subprocess.run(['git','-C','/tmp/keiba_clone','sparse-checkout','set','src'], check=True)
subprocess.run(['git','-C','/tmp/keiba_clone','checkout'], check=True)

if os.path.exists(SRC_DIR):
    shutil.rmtree(SRC_DIR)
shutil.copytree('/tmp/keiba_clone/src', SRC_DIR)
print(f'✅ src/ を {BRANCH} ブランチから取得しました')
print(f'   {SRC_DIR}')

## セル3: エンジン初期化

In [ ]:
sys.path.insert(0, BASE_DIR)  # プロジェクトルートを追加

from src.features.engine import init_engine
init_engine(BASE_DIR)
print('準備完了')

## セル4: 重みチューニング

過去レース1着データから各因子の最適重みを計算し `data/optimal_weights.json` に保存します。

**所要時間**: データ数・試行回数による（目安 1〜3分）

In [ ]:
from src.tools.tune_weights import run_tuning

opt_weights = run_tuning(
    BASE_DIR,
    n_restarts=5,   # 多いほど精度↑・時間↑
    verbose=True,
)

## セル5: キャリブレーション

AI確率 vs 実際の勝率のズレを補正するキャリブレーターを学習し `data/calibrator.pkl` に保存します。

> ⚠ セル4（重みチューニング）の後に実行してください。

In [ ]:
from src.tools.calibrate import run_calibration

calibrator = run_calibration(
    BASE_DIR,
    method='isotonic',  # 'isotonic'（推奨）または 'platt'
    verbose=True,
)

## セル6: 乖離分析・診断レポート

- AI確率バケット別の実的中率（キャリブレーション精度確認）
- AI vs 市場乖離バケット別 ROI（妙味のある乖離帯の特定）
- 券種別・EV別 ROI

In [ ]:
from src.tools.analyze_divergence import run_analysis

stats = run_analysis(BASE_DIR)

## セル7: 反映確認

次回の土日/金曜ノートブック実行前に `init_engine(BASE_DIR)` を呼ぶだけで
最適重みとキャリブレーターが自動適用されます。

In [ ]:
# 確認: 最適重みとキャリブレーターが読み込まれるか
from src.features.engine import init_engine
init_engine(BASE_DIR)

import json, os
w_path = f'{BASE_DIR}/data/optimal_weights.json'
c_path = f'{BASE_DIR}/data/calibrator.pkl'
print(f'optimal_weights.json: {"✅" if os.path.exists(w_path) else "❌"}')
print(f'calibrator.pkl:       {"✅" if os.path.exists(c_path) else "❌"}')

if os.path.exists(w_path):
    with open(w_path) as f: w = json.load(f)
    print('\n最適重み:')
    for k,v in sorted({k:v for k,v in w.items() if not k.startswith('_')}.items(), key=lambda x:-x[1]):
        print(f'  {k:10s}: {v:.4f}  {"█"*int(v*40)}')